# Positron Atlas — Python API

Drive an Atlas panel programmatically from a notebook cell.
The `positron_atlas` module is injected into `sys.modules` by the extension;
nothing is added to your namespace until you call `atlas.open(df)`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

# A tiny synthetic dataset with three clusters and some numeric/categorical columns.
centers = np.array([[0, 0], [5, 2], [2, 6]])
labels = rng.integers(0, 3, n)
noise = rng.normal(0, 0.8, (n, 2))

df = pd.DataFrame({
    **{f'feat_{i}': rng.normal(0, 1, n) + centers[labels, i % 2] for i in range(16)},
    'cluster': [f'Group {c}' for c in labels],
    'score': rng.uniform(0, 1, n).round(3),
    'label': [f'Row {i}' for i in range(n)],
})

In [ ]:
# Open an Atlas panel for `df`.
# Returns a handle immediately — no waiting for the UMAP to finish.
import positron_atlas as atlas

m = atlas.open(df)
m

In [ ]:
# Color by the 'cluster' column and label the map.
# These are fire-and-forget: the commands are sent and the cell returns at once.
m.color_by('cluster')
m.label_by('cluster')

In [ ]:
# Switch to 3D.
m.dims = 3

In [ ]:
# Add an annotation to a specific row.
m.annotate(row=0, text='Row zero')

In [ ]:
# Register a callback that fires when the lasso selection changes.
# The callback runs between cell executions (not while a cell is running).

def on_selection(rows):
    print(f'{len(rows)} rows selected: {rows[:5]}...' if len(rows) > 5 else f'{len(rows)} rows selected: {rows}')

m.on_select(on_selection)
print('Selection callback registered. Use the lasso tool in Atlas, then run the next cell.')

In [ ]:
# After drawing a lasso selection in Atlas, read the selection here.
# selected_rows returns a numpy array of row indices; selection returns a filtered DataFrame.
print('Selected row indices:', m.selected_rows)
print('Selected DataFrame:')
m.selection

In [ ]:
# Save the current map view as a PNG.
# save_png blocks until the host writes the file (up to 60 s).
import pathlib, tempfile

png_path = pathlib.Path(tempfile.mktemp(suffix='.png'))
m.save_png(str(png_path))
print(f'Saved {png_path.stat().st_size:,} bytes to {png_path}')

In [ ]:
# Export a self-contained interactive HTML file.
html_path = pathlib.Path(tempfile.mktemp(suffix='.html'))
m.save_html(str(html_path))
print(f'Saved {html_path.stat().st_size:,} bytes to {html_path}')

In [ ]:
# Close the Atlas panel.
m.close()